# NB11 — Text and Sentiment
**Machine Learning in Finance · Week 11**  [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ruiyuan-chen/ml-finance-course/blob/main/notebooks/NB11_text_and_sentiment.ipynb)

**Finance question:** Can a machine read the news and tell whether it's good or bad?

**Key idea:** Text becomes numbers. The simplest way **counts** positive and negative words from a finance dictionary. A better way **learns** which words matter from labelled examples. Either score could then become one more feature in a stock model.

In [ ]:
# ▶ RUN ME FIRST — loads the course tools and data (about 30 seconds the first time)
COURSE_URL = "https://raw.githubusercontent.com/ruiyuan-chen/ml-finance-course/main/"
import os, sys, urllib.request
here = [f for f in [".", "..", "../.."] if os.path.exists(os.path.join(f, "mlfin.py"))]
if here:
    sys.path.insert(0, here[0])
else:
    urllib.request.urlretrieve(COURSE_URL + "mlfin.py", "mlfin.py")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import mlfin
mlfin.setup(COURSE_URL)

## Step 1: meet the headlines
About 4,800 sentences from financial news, each labelled positive, neutral or negative by a team of
finance-trained readers (the *Financial PhraseBank*). `agreement` is the share of readers who agreed.

In [ ]:
heads = mlfin.load_headlines()
print(heads["label"].value_counts(normalize=True).round(3))
heads.sample(6, random_state=3)[["text", "label", "agreement"]]

## Step 2: score text with a finance dictionary
The **Loughran–McDonald** dictionary lists words that sound positive or negative *in finance*
("liability" is not negative in a 10-K; "crude" is a commodity, not an insult). The score is
(positive − negative) ÷ (positive + negative): from −1 to +1, and 0 if no listed word appears.

In [ ]:
# 🟡 YOUR TURN
TEXT = "The company reported strong growth, but warned of losses in its European unit."   # ← write your own

print("score:", mlfin.sentiment_dictionary(TEXT))
print("label:", mlfin.dictionary_label(TEXT))
mlfin.dictionary_words(TEXT)

## Step 3: where the dictionary goes wrong
Headlines where **every** reader agreed, but the dictionary didn't. Look at the words it found (or
didn't). What pattern do you see?

In [ ]:
# 🟡 YOUR TURN
SEED = 1   # ← change to see a different set of mistakes

mlfin.dictionary_mistakes(n=10, seed=SEED)

*Hint:* look for "rose", "fell", "increased", "decreased". Are those in the dictionary?

## Step 4: let the machine learn which words matter
Split the headlines: the classifier learns from 75% and is graded on the 25% it never saw. It turns each
sentence into word counts (**TF-IDF**: common-everywhere words count less) and fits a logistic regression
(Week 5) with three classes. We grade three methods on the same unseen headlines.

In [ ]:
# 🟡 YOUR TURN
VOCAB = 2000   # ← how many words and two-word phrases the classifier may use: try 50, 500, None (all)

accuracy, clf, test = mlfin.text_experiment(vocab_size=VOCAB)
accuracy

In [ ]:
print("Dictionary:")
display(mlfin.text_confusion(test))
print("Trained classifier:")
mlfin.text_confusion(test, clf)

*How to read the tables:* rows are what the human readers said, columns what the method said.
The diagonal is "right". Where does each method make most of its mistakes?

## Step 5: what did it learn?
The words that push the classifier hardest toward each label.

In [ ]:
mlfin.plot_top_words(clf, n=12)

## Step 6: your own headlines
Write headlines that should fool one method or the other: sarcasm, double negatives, "less bad than
expected", jargon.

In [ ]:
# 🟡 YOUR TURN
MY_HEADLINES = [
    "Profit fell less than analysts feared",
    "Company beats lowered expectations",
    "Shares plunge after CEO resigns",
    "The bank raised its dividend by 10 percent",
    # ← add three of your own
]
mlfin.score_headlines(MY_HEADLINES, clf)

## Step 7: more words, better model?
The Week 4 lesson again: more inputs can fit the training data better and the new data worse.

In [ ]:
mlfin.vocab_experiment(sizes=[50, 100, 500, 2000, None])

## Step 8: from a score to a trading feature
To use sentiment in the stock model you would need, for every stock and month: the news about **that
company**, time-stamped, scored, then averaged over the month *before* the prediction date. Our headlines
have no tickers or dates, so we stop here. Two warnings if you try it with real news:

* **Timing:** a story published at 4:05 pm can't be traded at that day's close. Use only news published
  before the moment you trade.
* **Who else knows:** if the whole market reads the same headline in milliseconds, the price may move
  before you can act.

## ⭐ GOING FURTHER (optional)
1. Copy the ten headlines from Step 3 into an AI assistant and ask it to label each one positive,
   neutral or negative. How many does it get right? Does it beat the classifier on the tricky ones?
2. Only learn from headlines all readers agreed on: `h = mlfin.load_headlines(); h[h["agreement"] == 100]`.
   Is the classifier more accurate?
3. Ask an AI assistant: *"What is the difference between a sentiment dictionary and a large language
   model when reading an earnings call?"*

## 🪞 REFLECTION (graded — write in your own words, no AI)

Double-click this cell and replace the *...* with your answers.

1. **In one sentence, what finance question did this week's technique answer?**
   *...*
2. **What is one way the result could be misleading?**
   *...*
3. **What did you ask an AI assistant this week, and was its answer right?**
   *...*